In [18]:
from pathlib import Path
from PIL import Image
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

In [21]:
class CaptchaDataset(Dataset):
    def __init__(self, image_dir, csv_path, transform=None, max_length=10):
        # Store dataset settings
        self.image_dir = Path(image_dir)
        self.csv_path = Path(csv_path)
        self.transform = transform
        self.max_length = max_length

        # Load CSV once
        self.labels_df = pd.read_csv(self.csv_path)

        # Adjust these column names if your CSV uses different names
        self.image_names = self.labels_df["filename"].tolist()
        self.labels = self.labels_df["label"].tolist()

        # Character vocabulary + blank token for padding
        self.chars = "2346789ABCDEFGHJKLMNPQRTUVWXYabcdefghijkmnpqrtuvwxy"
        self.blank_token = "<BLANK>"

        # Build lookup tables
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.char_to_idx[self.blank_token] = len(self.chars)
        self.idx_to_char = {i: c for c, i in self.char_to_idx.items()}

    def __len__(self):
        # Number of samples
        return len(self.image_names)

    def encode_label(self, label):
        # Convert text label into integer indices
        encoded = [self.char_to_idx[c] for c in label]

        # Pad to fixed length with blank token
        while len(encoded) < self.max_length:
            encoded.append(self.char_to_idx[self.blank_token])

        return torch.tensor(encoded, dtype=torch.long)

    def decode_label(self, encoded_tensor):
        # Convert encoded label back to readable text
        blank_idx = self.char_to_idx[self.blank_token]
        chars = [self.idx_to_char[int(idx)] for idx in encoded_tensor if int(idx) != blank_idx]
        return "".join(chars)

    def __getitem__(self, idx):
        # Get one filename + raw text label
        img_name = self.image_names[idx]
        label = self.labels[idx]

        # Load one grayscale image
        img_path = self.image_dir / img_name
        image = Image.open(img_path).convert("L")

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        # Encode label into tensor
        label_tensor = self.encode_label(label)

        return image, label_tensor

In [22]:
from pathlib import Path

image_dir = Path("data/processed/4Char_2000_CapGen_grayscale")
csv_path = image_dir / "ground_truth_index.csv"

print("Image folder exists:", image_dir.exists())
print("CSV exists:", csv_path.exists())

if image_dir.exists():
    print("Some files in folder:")
    for f in list(image_dir.iterdir())[:10]:
        print(" ", f.name)

Image folder exists: False
CSV exists: False
